In [3]:
# Install owlready2 if not installed
# !pip install owlready2

# Import libraries
from owlready2 import *

In [4]:
# Step 1: Load the Ontology
print("Loading ontology...")
onto = get_ontology("DrugInteraction.owl").load()
print("Ontology loaded successfully.\n")

Loading ontology...
Ontology loaded successfully.



In [5]:
# Step 2: Explore Ontology Structure
print("Classes in the ontology:")
for cls in onto.classes():
    print(cls)

print("\nObject Properties in the ontology:")
for prop in onto.object_properties():
    print(prop)

print("\nInstances of classes:")
for ind in onto.individuals():
    print(f"{ind} is an instance of {ind.is_a[0]}")

Classes in the ontology:
DrugInteraction.TestProgram
DrugInteraction.Drug
DrugInteraction.Interaction
DrugInteraction.Conflict
DrugInteraction.Similarity

Object Properties in the ontology:
DrugInteraction.causedBy
DrugInteraction.between
DrugInteraction.relatedTo

Instances of classes:
DrugInteraction.Warfarin is an instance of DrugInteraction.Drug
DrugInteraction.Aspirin is an instance of DrugInteraction.Drug
DrugInteraction.Nitroglycerin is an instance of DrugInteraction.Drug
DrugInteraction.Sildenafil is an instance of DrugInteraction.Drug
DrugInteraction.Interaction1 is an instance of DrugInteraction.Interaction
DrugInteraction.Conflict1 is an instance of DrugInteraction.Conflict
DrugInteraction.Similarity1 is an instance of DrugInteraction.Similarity


In [6]:
# Step 3: Query Relationships
print("\nQuerying Drug Interactions:")
interaction_class = onto.search_one(iri="*Interaction")
for interaction in interaction_class.instances():
    caused_by_drugs = [drug.name for drug in interaction.causedBy]
    print(f"Interaction: {interaction.name}, Caused By: {', '.join(caused_by_drugs)}")
    



Querying Drug Interactions:
Interaction: Interaction1, Caused By: Warfarin, Aspirin


In [7]:
print("\nQuerying Conflicts:") 
conflict_class = onto.search_one(iri="*Conflict") 
for conflict in conflict_class.instances(): 
	print(f"Conflict: {conflict.name}, Between: {conflict.between}") 
	involved_drugs = [] 
	for drug in conflict.between: 
		print(f"Drug: {drug}") # Print the drug to see its structure 
		try: 
			involved_drugs.append(drug.name) 
		except AttributeError: 
			print(f"Error: 'str' object has no attribute 'name'. Drug: {drug}") 		
print(f"Conflict: {conflict.name}, Involves Drugs: {', '.join(involved_drugs)}")



Querying Conflicts:
Conflict: Conflict1, Between: [DrugInteraction.Sildenafil, DrugInteraction.Nitroglycerin]
Drug: DrugInteraction.Sildenafil
Drug: DrugInteraction.Nitroglycerin
Conflict: Conflict1, Involves Drugs: Sildenafil, Nitroglycerin


In [8]:
# Querying Similarities with error handling
print("\nQuerying Similarities:")
similarity_class = onto.search_one(iri="*Similarity")

if similarity_class:  # Ensure the class exists
    for similarity in similarity_class.instances():
        # Safely handle 'relatedTo' to avoid AttributeError
        related_drugs = [
            drug.name if isinstance(drug, Thing) else str(drug) 
            for drug in getattr(similarity, "relatedTo", [])
        ]
        print(f"Similarity: {similarity.name}, Related Drugs: {', '.join(related_drugs)}")
else:
    print("Similarity class not found in the ontology.")



Querying Similarities:
Similarity: Similarity1, Related Drugs: Warfarin, Aspirin


In [9]:
# Step 4: Test Reasoning
# Add reasoning to infer new facts based on ontology
with onto:
    sync_reasoner()  # Runs the HermiT reasoner
print("\nReasoning complete. Checking inferred facts...")

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /home/manith/.local/lib/python3.12/site-packages/owlready2/hermit:/home/manith/.local/lib/python3.12/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmps4u6q73_



Reasoning complete. Checking inferred facts...


* Owlready2 * HermiT took 0.49095606803894043 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [10]:
# Re-query after reasoning
print("\nInferred Relationships for Interactions:")
for interaction in interaction_class.instances():
    caused_by_drugs = [drug.name for drug in interaction.causedBy]
    print(f"Interaction: {interaction.name}, Caused By: {', '.join(caused_by_drugs)}")


Inferred Relationships for Interactions:
Interaction: Interaction1, Caused By: Warfarin, Aspirin


In [11]:
# Step 5: Custom Queries and Testing
print("\nCustom Query: Find interactions involving 'Warfarin'")
warfarin = onto.search_one(iri="*Warfarin")
for interaction in interaction_class.instances():
    if warfarin in interaction.causedBy:
        print(f"Interaction Found: {interaction.name} caused by Warfarin")


Custom Query: Find interactions involving 'Warfarin'
Interaction Found: Interaction1 caused by Warfarin


In [12]:
print("\nCustom Query: Find conflicts involving 'Nitroglycerin'")

# Dynamically find the Nitroglycerin individual
nitroglycerin = onto.search_one(iri="*Nitroglycerin")

if nitroglycerin:
    # Query Conflicts and check 'between' property
    for conflict in conflict_class.instances():
        involved_drugs = getattr(conflict, "between", [])
        if nitroglycerin in involved_drugs:
            print(f"Conflict Found: {conflict.name} involving Nitroglycerin")
else:
    print("Nitroglycerin not found in the ontology.")



Custom Query: Find conflicts involving 'Nitroglycerin'
Conflict Found: Conflict1 involving Nitroglycerin
